In [1]:
import numpy  as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df = pd.read_csv("train.txt", sep=';', header=None, names=['text','emotion'])
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [3]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [4]:
unique_emotions = df['emotion'].unique()

In [5]:
emotion_numbers = {}
i = 0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i += 1
df['emotion'] = df['emotion'].map(emotion_numbers)

In [6]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [7]:
df['text'] = df['text'].apply(lambda x:x.lower())

In [8]:
import string 
def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

In [9]:
df['text'] = df['text'].apply(remove_punc)

In [10]:
def remove_numbers(txt):
    new =''
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new 

df['text'] = df['text'].apply(remove_numbers)

In [11]:
def remove_emojis(txt):
    new = ''
    for i in txt:
        if i.isascii():
            new += i 
    return new 

df['text'] = df['text'].apply(remove_emojis)

In [12]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [13]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [14]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [15]:
def remove(txt):
    words = word_tokenize(txt) 
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

In [16]:
df['text'] = df['text'].apply(remove)

In [17]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [18]:
from sklearn.feature_extraction.text import CountVectorizer

documents ={
    "I love pizza",
    "Pizza is the best",
    "I love pasta", 
    "Pasta is great"
}

# vectorizer = CountVectorizer()
vectorizer = CountVectorizer(ngram_range=(2,2))

X= vectorizer.fit_transform(documents)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("\n BoW Matrix:\n", X.toarray()) #bage of word

Vocabulary: ['is great' 'is the' 'love pasta' 'love pizza' 'pasta is' 'pizza is'
 'the best']

 BoW Matrix:
 [[0 0 0 1 0 0 0]
 [0 1 0 0 0 1 1]
 [0 0 1 0 0 0 0]
 [1 0 0 0 1 0 0]]


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents ={
    "I love pizza",
    "Pizza is the best",
    "I love pasta", 
    "Pasta is great"
}

# vectorizer = CountVectorizer()
vectorizer = TfidfVectorizer()

X= vectorizer.fit_transform(documents)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("\n BoW Matrix:\n", X.toarray()) #bage of word

Vocabulary: ['best' 'great' 'is' 'love' 'pasta' 'pizza' 'the']

 BoW Matrix:
 [[0.         0.         0.         0.70710678 0.         0.70710678
  0.        ]
 [0.55528266 0.         0.43779123 0.         0.         0.43779123
  0.55528266]
 [0.         0.         0.         0.70710678 0.70710678 0.
  0.        ]
 [0.         0.66767854 0.52640543 0.         0.52640543 0.
  0.        ]]


In [20]:
df

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
15995,brief time beanbag said anna feel like beaten,0
15996,turning feel pathetic still waiting tables sub...,0
15997,feel strong good overall,5
15998,feel like rude comment im glad,1


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [27]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,mean_absolute_error

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))
print(mean_absolute_error(y_test, pred_bow))

0.7678125
0.6340625


In [28]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0], shape=(3200,))

In [30]:
y_test

8756     0
4660     5
6095     0
304      5
8241     0
        ..
15578    5
5746     5
6395     5
7624     5
15245    0
Name: emotion, Length: 3200, dtype: int64

In [32]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

y_pred = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred))

0.6609375


In [34]:
from sklearn.linear_model import LogisticRegression
     
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf,y_train)
     
LogisticRegression(max_iter=1000)

log_pred = logistic_model.predict(X_test_tfidf)
     
print(accuracy_score(y_test,log_pred))

0.8615625
